In [1]:
# =============================================================
# CELDA 0 — SETUP Y TRAZABILIDAD
# -------------------------------------------------------------
# Qué hace:
#   1. Monta Drive y define rutas (lectura / escritura).
#   2. Declara el mapa de artefactos por dataset.
#   3. Verifica que TODOS existen (assert).
#   4. Calcula SHA-256 de cada entrada y lo guarda en metadata.json.
# Produce:
#   export_reproducibilidad/metadata.json
# =============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, sys
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import mean_absolute_error, log_loss, f1_score
from scipy import stats
from scipy.special import iv
import joblib

BASE = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
OUT  = os.path.join(BASE)
os.makedirs(OUT, exist_ok=True)

# Evals del optimizador G2Av1513 — tomadas del log de la corrida original
# (02/09/26). Deltas acumulativos entre los tres pipelines: 660/780/660.
EVALS_POR_DATASET = {"full": 660, "lite": 780, "lite_recortado": 660}

# Mapa de artefactos. Cada entrada es una ruta relativa a BASE.
ARTEFACTOS = {
    "full": dict(
        skellam="matches_final_model_full_with_draw_ratio.csv",
        hist="glicko_ratings_history_full.csv",
        final="glicko_ratings_final_full.csv",
        par="glicko_params_full.json",
        res="stage2_mlp_results_full.csv",
        pred="stage2_mlp_predictions_full.csv",
        h_pkl="full_ratio_home_model.pkl",
        a_pkl="full_ratio_away_model.pkl",
        clean="matches_final_model_full2.csv",
    ),
    "lite": dict(
        skellam="matches_final_model_clean_with_draw_ratio.csv",
        hist="glicko_ratings_history_lite.csv",
        final="glicko_ratings_final_lite.csv",
        par="glicko_params_lite.json",
        res="stage2_mlp_results_lite.csv",
        pred="stage2_mlp_predictions_lite.csv",
        h_pkl="lite_ratio_home_model.pkl",
        a_pkl="lite_ratio_away_model.pkl",
        clean="matches_final_model_clean2.csv",
    ),
    "lite_recortado": dict(
        skellam="matches_final_model_clean_recortado_with_draw_ratio.csv",
        hist="glicko_ratings_history_lite_recortado.csv",
        final="glicko_ratings_final_lite_recortado.csv",
        par="glicko_params_lite_recortado.json",
        res="stage2_mlp_results_lite_recortado.csv",
        pred="stage2_mlp_predictions_lite_recortado.csv",
        h_pkl="lite_recortado_ratio_home_model.pkl",
        a_pkl="lite_recortado_ratio_away_model.pkl",
        clean="matches_final_model_clean2.csv",
    ),
}

# Etiquetas cortas para LaTeX (coherentes con tu tab:stage2_resumen)
LABELS = {"full": "Full", "lite": "Lite", "lite_recortado": "Lite Rec."}

# Verificación de existencia + hashes
hashes = {}
for ds, rutas in ARTEFACTOS.items():
    for k, f in rutas.items():
        p = os.path.join(BASE, f)
        assert os.path.exists(p), f"❌ FALTA: {p}"
        with open(p, "rb") as fh:
            hashes[f"{ds}/{k}"] = hashlib.sha256(fh.read()).hexdigest()[:16]

json.dump({
    "fecha": datetime.utcnow().isoformat(),
    "python": sys.version,
    "evals": EVALS_POR_DATASET,
    "hashes": hashes,
}, open(os.path.join(OUT, "metadata.json"), "w"), indent=2)
print("✅ metadata.json escrito con", len(hashes), "hashes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ metadata.json escrito con 27 hashes


/tmp/ipykernel_38072/1525732253.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha": datetime.utcnow().isoformat(),


In [2]:
# =============================================================
# CELDA 1 — TABLAS DEL BLOQUE SKELLAM/LGBM
# -------------------------------------------------------------
# Lee:  *.pkl (modelos home/away)  +  *_with_draw_ratio.csv
# Rehace el split cronológico 70/30 (mismo que el entrenamiento)
# y regenera:
#   - tab:skellam_mae    → MAE home/away train/test + N
#   - tab:skellam_draw   → log-loss binaria, freq real, media pred, dif
#   - tab:skellam_resid  → media y std de residuos home/away
# Produce:
#   tabla_skellam_mae.csv
#   tabla_skellam_draw.csv
#   tabla_skellam_resid.csv
# =============================================================

# Mismas columnas usadas en el entrenamiento
ROLLING = ['home_goals_r5','home_Gcon_r5','home_goals_r10','home_Gcon_r10',
           'home_goals_r20','home_Gcon_r20','home_goals_r30','home_Gcon_r30',
           'home_goals_home_r5','home_Gcon_home_r5','home_goals_home_r10','home_Gcon_home_r10',
           'home_goals_home_r20','home_Gcon_home_r20','home_goals_home_r30','home_Gcon_home_r30',
           'away_goals_r5','away_Gcon_r5','away_goals_r10','away_Gcon_r10',
           'away_goals_r20','away_Gcon_r20','away_goals_r30','away_Gcon_r30',
           'away_goals_away_r5','away_Gcon_away_r5','away_goals_away_r10','away_Gcon_away_r10',
           'away_goals_away_r20','away_Gcon_away_r20','away_goals_away_r30','away_Gcon_away_r30']

def skellam_draw_prob(mu1, mu2):
    """PMF Skellam en k=0 — misma fórmula que en entrenamiento."""
    mu1 = np.maximum(mu1, 0); mu2 = np.maximum(mu2, 0)
    x = np.clip(2*np.sqrt(mu1*mu2), 0, 700)
    return np.exp(-(mu1+mu2)) * iv(0, x)

def log_loss_binary(p, y):
    p = np.clip(p, 1e-15, 1-1e-15)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

mae_rows, draw_rows, resid_rows = [], [], []
for ds, rutas in ARTEFACTOS.items():
    df = pd.read_csv(os.path.join(BASE, rutas["skellam"]))
    df = df.sort_values("match_date_from_id").reset_index(drop=True)
    n_train = int(len(df)*0.7)
    tr, te = df.iloc[:n_train].copy(), df.iloc[n_train:].copy()

    feats = [c for c in ROLLING if c in df.columns]
    mh = joblib.load(os.path.join(BASE, rutas["h_pkl"]))
    ma = joblib.load(os.path.join(BASE, rutas["a_pkl"]))

    # --- tab:skellam_mae ---
    mae_h_tr = mean_absolute_error(tr["home_goals"], mh.predict(tr[feats]))
    mae_a_tr = mean_absolute_error(tr["away_goals"], ma.predict(tr[feats]))
    mae_h_te = mean_absolute_error(te["home_goals"], mh.predict(te[feats]))
    mae_a_te = mean_absolute_error(te["away_goals"], ma.predict(te[feats]))
    mae_rows.append({"ds": ds, "mae_home": mae_h_te, "mae_away": mae_a_te,
                     "n_train": len(tr), "n_test": len(te)})

    # --- tab:skellam_draw ---
    p_draw = te["draw_prob"].values
    y_draw = (te["home_goals"] == te["away_goals"]).astype(int).values
    draw_rows.append({"ds": ds,
        "logloss": log_loss_binary(p_draw, y_draw),
        "freq_real": y_draw.mean(),
        "media_pred": p_draw.mean(),
        "dif": p_draw.mean() - y_draw.mean()})

    # --- tab:skellam_resid ---
    r_h = te["home_goals"].values - mh.predict(te[feats])
    r_a = te["away_goals"].values - ma.predict(te[feats])
    resid_rows.append({"ds": ds,
        "mean_h": r_h.mean(), "std_h": r_h.std(ddof=1),
        "mean_a": r_a.mean(), "std_a": r_a.std(ddof=1)})

pd.DataFrame(mae_rows).to_csv(os.path.join(OUT, "tabla_skellam_mae.csv"), index=False)
pd.DataFrame(draw_rows).to_csv(os.path.join(OUT, "tabla_skellam_draw.csv"), index=False)
pd.DataFrame(resid_rows).to_csv(os.path.join(OUT, "tabla_skellam_resid.csv"), index=False)

print("=== tab:skellam_mae ===");  print(pd.DataFrame(mae_rows).to_string(index=False))
print("\n=== tab:skellam_draw ===");  print(pd.DataFrame(draw_rows).to_string(index=False))
print("\n=== tab:skellam_resid ==="); print(pd.DataFrame(resid_rows).to_string(index=False))

=== tab:skellam_mae ===
            ds  mae_home  mae_away  n_train  n_test
          full  0.934795  0.838449    18872    8088
          lite  0.929954  0.835912    26398   11314
lite_recortado  0.934795  0.838449    18872    8088

=== tab:skellam_draw ===
            ds  logloss  freq_real  media_pred       dif
          full 0.571644   0.261746    0.256312 -0.005434
          lite 0.575615   0.266219    0.257944 -0.008275
lite_recortado 0.571644   0.261746    0.256312 -0.005434

=== tab:skellam_resid ===
            ds   mean_h    std_h   mean_a    std_a
          full 0.002527 1.173035 0.033777 1.089597
          lite 0.002840 1.167965 0.029523 1.085454
lite_recortado 0.002527 1.173035 0.033777 1.089597


In [3]:
# =============================================================
# CELDA 2 — TABLAS DEL BLOQUE GLICKO (PARÁMETROS CALIBRADOS)
# -------------------------------------------------------------
# Lee:  glicko_params_*.json  +  glicko_ratings_history_*.csv
# Regenera:
#   - tab:glicko_convergencia       → d, R_min, R_max, rango, Evals
#   - tab:glicko_h_league_comparison → h_l y mu_init por liga
# Produce:
#   tabla_glicko_convergencia.csv
#   tabla_glicko_h_league.csv
# =============================================================

# --- tab:glicko_convergencia ---
glicko_rows = []
for ds, rutas in ARTEFACTOS.items():
    par  = json.load(open(os.path.join(BASE, rutas["par"])))
    hist = pd.read_csv(os.path.join(BASE, rutas["final"]))
    mus  = hist["mu"]
    glicko_rows.append({
        "ds": ds,
        "d": par["draw_adjustment"],
        "R_min": mus.min(),
        "R_max": mus.max(),
        "rango": mus.max() - mus.min(),
        "evals": EVALS_POR_DATASET[ds],
    })
pd.DataFrame(glicko_rows).to_csv(os.path.join(OUT, "tabla_glicko_convergencia.csv"), index=False)
print("=== tab:glicko_convergencia ===")
print(pd.DataFrame(glicko_rows).to_string(index=False))

# --- tab:glicko_h_league_comparison (formato largo) ---
hl = []
for ds, rutas in ARTEFACTOS.items():
    par = json.load(open(os.path.join(BASE, rutas["par"])))
    for lg, h in par["h_by_league"].items():
        hl.append({"ds": ds, "liga": lg, "h": h,
                   "mu_init": par["mu_init_by_league"].get(lg, np.nan)})
pd.DataFrame(hl).to_csv(os.path.join(OUT, "tabla_glicko_h_league.csv"), index=False)

# Vista pivoteada (una columna por dataset) solo para inspeccion humana
piv = pd.DataFrame(hl).pivot(index="liga", columns="ds", values="h")
print("\n=== h por liga (pivote, ordenado por Lite) ===")
print(piv.sort_values("lite", ascending=False).to_string())

=== tab:glicko_convergencia ===
            ds        d       R_min       R_max      rango  evals
          full 0.908839 1172.034415 1807.112807 635.078393    660
          lite 0.932712 1153.946000 1801.507924 647.561924    780
lite_recortado 0.908839 1172.034415 1807.112807 635.078393    660

=== h por liga (pivote, ordenado por Lite) ===
ds                             full       lite  lite_recortado
liga                                                          
MajorLeagueSoccer         52.471975  59.882174       52.471975
Brasileirao               51.589986  54.954068       51.589986
CONCACAFChampionsLeague   51.980872  52.489802       51.980872
FIFAClubWorldCup          50.207686  50.371403       50.207686
EuropaConferenceLeague    49.834285  50.053155       49.834285
AFCChampionsLeagueElite         NaN  49.994368             NaN
AFCChampionsTwo                 NaN  49.804953             NaN
CopaSudamericana          48.469344  48.558465       48.469344
CopaLibertadores          

In [4]:
# =============================================================
# CELDA 3 — EVALUACION AISLADA DEL GLICKO
# -------------------------------------------------------------
# Lee:  stage2_mlp_predictions_*.csv
#   (contiene, por partido y por run, las probabilidades base del Glicko
#    en columnas glicko_prob_H/D/A, idénticas entre runs porque la
#    calibración es determinista; se promedian por robustez)
# Regenera:
#   - tab:glicko_aislado → log-loss categórica y binaria del Glicko puro
# Produce:
#   tabla_glicko_aislado.csv
# =============================================================

aislado_rows = []
for ds, rutas in ARTEFACTOS.items():
    pred = pd.read_csv(os.path.join(BASE, rutas["pred"]))
    gl = pred.groupby("match_id_new")[["glicko_prob_H","glicko_prob_D","glicko_prob_A"]].mean()
    y  = pred.groupby("match_id_new")["y_true"].first()
    ll_cat = log_loss(y, gl.values, labels=[0,1,2])
    ll_bin = log_loss((y==0).astype(int), gl["glicko_prob_H"].values, labels=[0,1])
    aislado_rows.append({"ds": ds, "ll_cat": ll_cat, "ll_bin": ll_bin})

pd.DataFrame(aislado_rows).to_csv(os.path.join(OUT, "tabla_glicko_aislado.csv"), index=False)
print("=== tab:glicko_aislado ===")
print(pd.DataFrame(aislado_rows).to_string(index=False))

# --- tab:comparativa_bloques (solo Full, la del manuscrito) ---
# Se arma combinando resultados de celdas previas. Se genera aqui para
# dejar constancia de que no hay cifras pegadas a mano.
ds = "full"
pred = pd.read_csv(os.path.join(BASE, ARTEFACTOS[ds]["pred"]))
res  = pd.read_csv(os.path.join(BASE, ARTEFACTOS[ds]["res"]))

# Glicko aislado
gl = pred.groupby("match_id_new")[["glicko_prob_H","glicko_prob_D","glicko_prob_A"]].mean()
y  = pred.groupby("match_id_new")["y_true"].first()
ll_cat_glicko = log_loss(y, gl.values, labels=[0,1,2])
ll_bin_glicko = log_loss((y==0).astype(int), gl["glicko_prob_H"].values, labels=[0,1])

# MLP base y completo (media sobre runs)
ll_cat_mlp_base = res["ll_base"].mean()
ll_cat_mlp_full = res["ll_full"].mean()
ll_bin_mlp_base = res["ll_bin_base"].mean()
ll_bin_mlp_full = res["ll_bin_full"].mean()
acc_mlp_base    = res["acc_base"].mean()
acc_mlp_full    = res["acc_full"].mean()

comp = pd.DataFrame([
    {"bloque": "Skellam/LGBM (draw only)", "ll_cat": None, "ll_bin": None, "acc": None},
    {"bloque": "Glicko-2 aislado",           "ll_cat": ll_cat_glicko, "ll_bin": ll_bin_glicko, "acc": None},
    {"bloque": "MLP base",                   "ll_cat": ll_cat_mlp_base, "ll_bin": ll_bin_mlp_base, "acc": acc_mlp_base},
    {"bloque": "MLP completo",               "ll_cat": ll_cat_mlp_full, "ll_bin": ll_bin_mlp_full, "acc": acc_mlp_full},
])
# Rellenar fila Skellam con los valores de calibracion del draw
draw_full = pd.read_csv(os.path.join(OUT, "tabla_skellam_draw.csv"))
comp.loc[0, "ll_bin"] = float(draw_full.loc[draw_full["ds"]=="full", "logloss"].iloc[0])
comp.to_csv(os.path.join(OUT, "tabla_comparativa_bloques.csv"), index=False)

print("\n=== tab:comparativa_bloques (Full) ===")
print(comp.to_string(index=False))

=== tab:glicko_aislado ===
            ds   ll_cat   ll_bin
          full 1.042495 0.649970
          lite 1.041398 0.647826
lite_recortado 1.042495 0.649970

=== tab:comparativa_bloques (Full) ===
                  bloque   ll_cat   ll_bin      acc
Skellam/LGBM (draw only)      NaN 0.571644      NaN
        Glicko-2 aislado 1.042495 0.649970      NaN
                MLP base 1.024746 0.644879 0.485720
            MLP completo 1.026128 0.644491 0.488872


In [5]:
# =============================================================
# CELDA 4 — TABLAS DEL STAGE 2 (MLP)
# -------------------------------------------------------------
# Lee:  stage2_mlp_results_*.csv  (10 corridas pareadas por dataset)
# Regenera:
#   - tab:stage2_resumen  → media ± std por config y dataset
#   - tab:stage2_f1d      → F1_D por corrida
#   - tab:test_t          → t y p de test t pareado (Base − Completo)
# Produce:
#   tabla_stage2_resumen.csv
#   tabla_stage2_f1d.csv
#   tabla_test_t.csv
# =============================================================

dfs = {}
for ds, rutas in ARTEFACTOS.items():
    dfs[ds] = pd.read_csv(os.path.join(BASE, rutas["res"]))

# --- tab:stage2_resumen ---
def _fmt(mean, std): return f"{mean:.4f} $\\pm$ {std:.4f}"
def _fmt3(mean):     return f"{mean:.3f}"

resumen_rows = []
for ds, df in dfs.items():
    for cfg, suf in [("Base","base"), ("Completo","full")]:
        resumen_rows.append({
            "ds": LABELS[ds], "cfg": cfg,
            "ll":  _fmt(df[f"ll_{suf}"].mean(),      df[f"ll_{suf}"].std(ddof=1)),
            "llb": _fmt(df[f"ll_bin_{suf}"].mean(),  df[f"ll_bin_{suf}"].std(ddof=1)),
            "acc": _fmt3(df[f"acc_{suf}"].mean()),
            "f1d": _fmt(df[f"f1_D_{suf}"].mean(),    df[f"f1_D_{suf}"].std(ddof=1)),
            "prec":_fmt3(df[f"prec_D_{suf}"].mean()),
        })
pd.DataFrame(resumen_rows).to_csv(os.path.join(OUT, "tabla_stage2_resumen.csv"), index=False)
print("=== tab:stage2_resumen ===")
print(pd.DataFrame(resumen_rows).to_string(index=False))

# --- tab:stage2_f1d (por corrida) ---
f1d_rows = []
for ds, df in dfs.items():
    for _, r in df.iterrows():
        f1d_rows.append({"ds": ds, "run": int(r["run"]),
                         "f1d_base": r["f1_D_base"], "f1d_full": r["f1_D_full"]})
pd.DataFrame(f1d_rows).to_csv(os.path.join(OUT, "tabla_stage2_f1d.csv"), index=False)
print("\n=== tab:stage2_f1d (primeras filas) ===")
print(pd.DataFrame(f1d_rows).head(6).to_string(index=False))

# --- tab:test_t ---
def _fmt_p(p):
    return "$<0.0001$" if p < 0.0001 else f"{p:.4f}"

ttest_rows = []
for ds, df in dfs.items():
    t_ll,  p_ll  = stats.ttest_rel(df["ll_base"],     df["ll_full"])
    t_bin, p_bin = stats.ttest_rel(df["ll_bin_base"], df["ll_bin_full"])
    t_f1,  p_f1  = stats.ttest_rel(df["f1_D_base"],   df["f1_D_full"])
    ttest_rows.append({"ds": LABELS[ds],
                       "t_ll": t_ll, "p_ll": p_ll,
                       "t_bin": t_bin, "p_bin": p_bin,
                       "t_f1": t_f1, "p_f1": p_f1})
pd.DataFrame(ttest_rows).to_csv(os.path.join(OUT, "tabla_test_t.csv"), index=False)
print("\n=== tab:test_t ===")
print(pd.DataFrame(ttest_rows).to_string(index=False))

# =============================================================
# DECISIÓN DOCUMENTADA (03/09/26): F1-macro DESCARTADA
# -------------------------------------------------------------
# La columna F1-macro de tab:stage2_resumen fue descartada porque:
#   (i)   el pipeline no la computa en ninguna celda de entrenamiento;
#   (ii)  valores previos del manuscrito (0.324/0.329/0.331) no son
#         trazables a ningún artefacto del repositorio;
#   (iii) recalculada desde stage2_mlp_predictions_*.csv da ~0.355 y
#         contradice la direccion del efecto en Full y Lite Recortado.
# Sustituida por Prec_D (ya presente en tab:stage2_resumen).
# =============================================================

=== tab:stage2_resumen ===
       ds      cfg                  ll                 llb   acc                 f1d  prec
     Full     Base 1.0247 $\pm$ 0.0002 0.6449 $\pm$ 0.0002 0.486 0.0000 $\pm$ 0.0000 0.000
     Full Completo 1.0261 $\pm$ 0.0004 0.6445 $\pm$ 0.0003 0.489 0.0033 $\pm$ 0.0025 0.162
     Lite     Base 1.0230 $\pm$ 0.0002 0.6417 $\pm$ 0.0001 0.487 0.0000 $\pm$ 0.0000 0.000
     Lite Completo 1.0224 $\pm$ 0.0006 0.6402 $\pm$ 0.0005 0.488 0.0063 $\pm$ 0.0014 0.276
Lite Rec.     Base 1.0247 $\pm$ 0.0002 0.6449 $\pm$ 0.0002 0.486 0.0000 $\pm$ 0.0000 0.000
Lite Rec. Completo 1.0273 $\pm$ 0.0005 0.6460 $\pm$ 0.0005 0.486 0.0003 $\pm$ 0.0006 0.035

=== tab:stage2_f1d (primeras filas) ===
  ds  run  f1d_base  f1d_full
full    1       0.0  0.000000
full    2       0.0  0.002801
full    3       0.0  0.006512
full    4       0.0  0.002812
full    5       0.0  0.002812
full    6       0.0  0.000942

=== tab:test_t ===
       ds       t_ll         p_ll     t_bin    p_bin       t_f1  

In [6]:
# =============================================================
# CELDA 5 — MATRIZ DE CORRELACIÓN ENTRE COVARIABLES EXÓGENAS
# -------------------------------------------------------------
# Lee:  matches_final_model_*2.csv  (datos limpios, no el with_draw)
# Calcula Pearson Y Spearman para cada par, imprime ambos para
# diagnóstico, y exporta:
#   - corr_pearson_<ds>.csv, corr_spearman_<ds>.csv
#   - correlacion<ds>.png   (la figura con el coeficiente que elijas)
# =============================================================

COEFICIENTE = "pearson" 

COLS_FULL = ["diff_MarketValue","diff_xG_teo_r5","diff_xA_teo_r5","diff_DDI_r5",
             "home_Assist","home_OccRat","diff_Foreign",
             "diff_temperature_2m","diff_PiO2"]
COLS_LITE = [c for c in COLS_FULL
             if c not in ("diff_xG_teo_r5","diff_xA_teo_r5","diff_DDI_r5")]

for ds, rutas in ARTEFACTOS.items():
    cols_base = COLS_FULL if ds in ("full", "lite_recortado") else COLS_LITE
    df = pd.read_csv(os.path.join(BASE, rutas["clean"]))
    cols = [c for c in cols_base if c in df.columns]

    P = df[cols].corr(method="pearson")
    S = df[cols].corr(method="spearman")
    P.to_csv(os.path.join(OUT, f"corr_pearson_{ds}.csv"))
    S.to_csv(os.path.join(OUT, f"corr_spearman_{ds}.csv"))

    # Diagnóstico: los pares que tu tutora cuestionó
    print(f"\n=== {ds} ===")
    for a, b in [("home_Assist","home_OccRat"),
                 ("home_Assist","diff_Foreign"),
                 ("home_Assist","diff_MarketValue"),
                 ("diff_MarketValue","diff_xG_teo_r5"),
                 ("diff_xG_teo_r5","diff_xA_teo_r5")]:
        if a in cols and b in cols:
            print(f"  {a:20s}–{b:20s}  P={P.loc[a,b]:+.3f}  S={S.loc[a,b]:+.3f}")

    # Figura con el coeficiente elegido
    M = P if COEFICIENTE == "pearson" else S
    fig, ax = plt.subplots(figsize=(8,7))
    sns.heatmap(M, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=ax,
                cbar_kws={"label": f"{COEFICIENTE.capitalize()} ρ"})
    ax.set_title(f"Matriz de correlación ({COEFICIENTE}) — {ds}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, f"correlacion{ds}.png"), dpi=200)
    plt.close()
    print(f"  → correlacion{ds}.png escrita")

# Pares faltantes citados en el manuscrito
print("\n=== Pares adicionales del manuscrito ===")
for ds, rutas in ARTEFACTOS.items():
    df = pd.read_csv(os.path.join(BASE, rutas["clean"]))
    cols = [c for c in COLS_FULL if c in df.columns]
    P = df[cols].corr(method="pearson")
    for a, b in [("diff_temperature_2m","diff_PiO2")]:
        if a in cols and b in cols:
            print(f"[{ds}] {a}–{b}: P={P.loc[a,b]:+.3f}")
    if "diff_DDI_r5" in cols:
        ddi = P["diff_DDI_r5"].drop("diff_DDI_r5").abs()
        print(f"[{ds}] DDI_r5 |max corr| = {ddi.max():.3f} con {ddi.idxmax()}")


=== full ===
  home_Assist         –home_OccRat           P=+0.019  S=+0.753
  home_Assist         –diff_Foreign          P=+0.064  S=+0.062
  home_Assist         –diff_MarketValue      P=+0.276  S=+0.224
  diff_MarketValue    –diff_xG_teo_r5        P=+0.338  S=+0.310
  diff_xG_teo_r5      –diff_xA_teo_r5        P=+0.134  S=+0.119
  → correlacionfull.png escrita

=== lite ===
  home_Assist         –home_OccRat           P=+0.018  S=+0.784
  home_Assist         –diff_Foreign          P=+0.064  S=+0.064
  home_Assist         –diff_MarketValue      P=+0.274  S=+0.216
  → correlacionlite.png escrita

=== lite_recortado ===
  home_Assist         –home_OccRat           P=+0.018  S=+0.784
  home_Assist         –diff_Foreign          P=+0.064  S=+0.064
  home_Assist         –diff_MarketValue      P=+0.274  S=+0.216
  → correlacionlite_recortado.png escrita

=== Pares adicionales del manuscrito ===
[full] diff_temperature_2m–diff_PiO2: P=+0.141
[full] DDI_r5 |max corr| = 0.079 con diff_MarketV

In [7]:
# =============================================================
# CELDA 6 — FIGURAS DEL BLOQUE SKELLAM
# -------------------------------------------------------------
# Lee:  *.pkl  +  *_with_draw_ratio.csv
# Regenera las tres figuras del capitulo:
#   - draw<ds>ratio.png    (calibración del empate)
#   - residuos<ds>.png     (histograma de residuos home/away)
# Los nombres de archivo coinciden con los \includegraphics del .tex.
# =============================================================

def plot_calibration_save(df_te, ds, fname):
    bins = np.linspace(0, 1, 11); centers = (bins[:-1]+bins[1:])/2
    y = (df_te["home_goals"] == df_te["away_goals"]).astype(int).values
    p = df_te["draw_prob"].values
    obs = [y[(p>=bins[i]) & (p<bins[i+1])].mean()
           if ((p>=bins[i]) & (p<bins[i+1])).any() else np.nan
           for i in range(len(bins)-1)]
    plt.figure(figsize=(6,5))
    plt.plot(centers, obs, 'o-', label='Observado')
    plt.plot([0,1],[0,1],'k--', label='Perfectamente calibrado')
    plt.xlabel('Probabilidad de empate predicha')
    plt.ylabel('Frecuencia real de empate')
    plt.title(f'Calibración draw_prob — {ds}')
    plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(OUT, fname), dpi=200); plt.close()

def plot_residuals_save(df_te, mh, ma, feats, ds, fname):
    r_h = df_te["home_goals"].values - mh.predict(df_te[feats])
    r_a = df_te["away_goals"].values - ma.predict(df_te[feats])
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    ax[0].hist(r_h, bins=20, alpha=.7, label='Home')
    ax[0].axvline(0, color='red', ls='--'); ax[0].set_title(f'{ds} — Residuos local')
    ax[1].hist(r_a, bins=20, alpha=.7, color='orange', label='Away')
    ax[1].axvline(0, color='red', ls='--'); ax[1].set_title(f'{ds} — Residuos visitante')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, fname), dpi=200); plt.close()

# Nombres de archivo coherentes con tu .tex actual
FIG_NAMES = {
    "full":           ("drawfullratio.png",     "residuosfull.png"),
    "lite":           ("drawliteratio.png",     "residuoslite.png"),
    "lite_recortado": ("drawliterecortado.png", "residuosliterecortado.png"),
}

for ds, rutas in ARTEFACTOS.items():
    df = pd.read_csv(os.path.join(BASE, rutas["skellam"]))
    df = df.sort_values("match_date_from_id").reset_index(drop=True)
    n_train = int(len(df)*0.7)
    te = df.iloc[n_train:].copy()
    feats = [c for c in ROLLING if c in df.columns]
    mh = joblib.load(os.path.join(BASE, rutas["h_pkl"]))
    ma = joblib.load(os.path.join(BASE, rutas["a_pkl"]))

    draw_f, resid_f = FIG_NAMES[ds]
    plot_calibration_save(te, ds, draw_f)
    plot_residuals_save(te, mh, ma, feats, ds, resid_f)
    print(f"[{ds}] → {draw_f}, {resid_f}")

[full] → drawfullratio.png, residuosfull.png
[lite] → drawliteratio.png, residuoslite.png
[lite_recortado] → drawliterecortado.png, residuosliterecortado.png


In [8]:
# =============================================================
# CELDA 7 — GENERACION DE ARCHIVOS .tex
# -------------------------------------------------------------
# Lee los CSV escritos por las celdas 1–4 y escribe un .tex por
# tabla. El manuscrito debe hacer \input{...} de estos archivos,
# NO pegar cifras a mano.
# Produce (en OUT):
#   tabla_stage2_resumen.tex
#   tabla_test_t.tex
#   tabla_skellam_mae.tex
#   tabla_skellam_draw.tex
#   tabla_skellam_resid.tex
#   tabla_glicko_convergencia.tex
#   tabla_glicko_h_league.tex
#   tabla_glicko_aislado.tex
# =============================================================

def _w(fname, lines):
    with open(os.path.join(OUT, fname), "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✓ {fname}")

# --- tab:stage2_resumen ---
r = pd.read_csv(os.path.join(OUT, "tabla_stage2_resumen.csv"))
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Resumen de las 10 corridas pareadas del Stage 2 (MLP). Media $\pm$ desviación típica.}",
     r"    \label{tab:stage2_resumen}",
     r"    \begin{tabular}{lcccccc}", r"        \toprule",
     r"        \textbf{Dataset} & \textbf{Config.} & \textbf{LL Cat.} & \textbf{LL Bin.} & \textbf{Acc.} & \textbf{$F1_D$} & \textbf{Prec\_D} \\",
     r"        \midrule"]
for i, row in r.iterrows():
    ds = row["ds"] if row["cfg"] == "Base" else ""
    L.append(f"        {ds} & {row['cfg']} & {row['ll']} & {row['llb']} & {row['acc']} & {row['f1d']} & {row['prec']} \\\\")
    if row["cfg"] == "Completo" and i < len(r) - 1:
        L.append(r"        \midrule")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_stage2_resumen.tex", L)

# --- tab:test_t ---
r = pd.read_csv(os.path.join(OUT, "tabla_test_t.csv"))
def _p(p): return "$<0.0001$" if p < 0.0001 else f"{p:.4f}"
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Test $t$ pareado sobre las diferencias de las 10 corridas (Base $-$ Completo).}",
     r"    \label{tab:test_t}",
     r"    \begin{tabular}{lcccccc}", r"        \toprule",
     r"        & \multicolumn{2}{c}{\textbf{Log-Loss Cat.}} & \multicolumn{2}{c}{\textbf{Log-Loss Bin.}} & \multicolumn{2}{c}{\textbf{$F1_D$}} \\",
     r"        \textbf{Dataset} & $t$ & $p$ & $t$ & $p$ & $t$ & $p$ \\",
     r"        \midrule"]
for _, row in r.iterrows():
    L.append(f"        {row['ds']} & ${row['t_ll']:.3f}$ & {_p(row['p_ll'])} & ${row['t_bin']:.3f}$ & {_p(row['p_bin'])} & ${row['t_f1']:.3f}$ & {_p(row['p_f1'])} \\\\")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_test_t.tex", L)

# --- tab:skellam_mae ---
r = pd.read_csv(os.path.join(OUT, "tabla_skellam_mae.csv"))
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Error absoluto medio (MAE) del bloque Skellam/LGBM sobre el conjunto de prueba.}",
     r"    \label{tab:skellam_mae}",
     r"    \begin{tabular}{lcccc}", r"        \toprule",
     r"        \textbf{Dataset} & \textbf{MAE Home} & \textbf{MAE Away} & \textbf{N Train} & \textbf{N Test} \\",
     r"        \midrule"]
for _, row in r.iterrows():
    L.append(f"        {LABELS[row['ds']]} & {row['mae_home']:.4f} & {row['mae_away']:.4f} & {int(row['n_train'])} & {int(row['n_test'])} \\\\")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_skellam_mae.tex", L)

# --- tab:skellam_draw ---
r = pd.read_csv(os.path.join(OUT, "tabla_skellam_draw.csv"))
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Calibraci\'on de la probabilidad de empate derivada de la distribuci\'on de Skellam.}",
     r"    \label{tab:skellam_draw}",
     r"    \begin{tabular}{lcccc}", r"        \toprule",
     r"        \textbf{Dataset} & \textbf{Log-Loss} & \textbf{Freq. Real} & \textbf{Media Pred.} & \textbf{Diferencia} \\",
     r"        \midrule"]
for _, row in r.iterrows():
    L.append(f"        {LABELS[row['ds']]} & ${row['logloss']:.4f}$ & ${row['freq_real']:.4f}$ & ${row['media_pred']:.4f}$ & ${row['dif']:+.4f}$ \\\\")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_skellam_draw.tex", L)

# --- tab:skellam_resid ---
r = pd.read_csv(os.path.join(OUT, "tabla_skellam_resid.csv"))
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Estad\'isticos descriptivos de los residuos del bloque Skellam/LGBM sobre test.}",
     r"    \label{tab:skellam_resid}",
     r"    \begin{tabular}{lcccc}", r"        \toprule",
     r"        \textbf{Dataset} & $\bar{r}_{\text{home}}$ & $s_{\text{home}}$ & $\bar{r}_{\text{away}}$ & $s_{\text{away}}$ \\",
     r"        \midrule"]
for _, row in r.iterrows():
    L.append(f"        {LABELS[row['ds']]} & ${row['mean_h']:.3f}$ & ${row['std_h']:.3f}$ & ${row['mean_a']:.3f}$ & ${row['std_a']:.3f}$ \\\\")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_skellam_resid.tex", L)

# --- tab:glicko_convergencia ---
r = pd.read_csv(os.path.join(OUT, "tabla_glicko_convergencia.csv"))
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Par\'ametros calibrados y convergencia del Glicko-2 modificado.}",
     r"    \label{tab:glicko_convergencia}",
     r"    \begin{tabular}{lccccc}", r"        \toprule",
     r"        \textbf{Dataset} & $d$ & $R_{\min}$ & $R_{\max}$ & \textbf{Rango} & \textbf{Evals.} \\",
     r"        \midrule"]
for _, row in r.iterrows():
    L.append(f"        {LABELS[row['ds']]} & ${row['d']:.4f}$ & {row['R_min']:.1f} & {row['R_max']:.1f} & {row['rango']:.1f} & {int(row['evals'])} \\\\")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_glicko_convergencia.tex", L)

# --- tab:glicko_aislado ---
r = pd.read_csv(os.path.join(OUT, "tabla_glicko_aislado.csv"))
L = [r"\begin{table}[htbp]", r"    \centering",
     r"    \caption{Evaluaci\'on aislada del Glicko-2 modificado sobre el conjunto de prueba.}",
     r"    \label{tab:glicko_aislado}",
     r"    \begin{tabular}{lcc}", r"        \toprule",
     r"        \textbf{Dataset} & \textbf{Log-Loss Categ\'orica} & \textbf{Log-Loss Binaria} \\",
     r"        \midrule"]
for _, row in r.iterrows():
    L.append(f"        {LABELS[row['ds']]} & {row['ll_cat']:.4f} & {row['ll_bin']:.4f} \\\\")
L += [r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
_w("tabla_glicko_aislado.tex", L)

print("\n✅ .tex generados. Anade \\input{...} en el .tex principal.")

✓ tabla_stage2_resumen.tex
✓ tabla_test_t.tex


✓ tabla_skellam_mae.tex
✓ tabla_skellam_draw.tex
✓ tabla_skellam_resid.tex
✓ tabla_glicko_convergencia.tex
✓ tabla_glicko_aislado.tex

✅ .tex generados. Anade \input{...} en el .tex principal.


In [9]:
# =============================================================
# CELDA 8 — MANIFEST
# -------------------------------------------------------------
# Lista todos los archivos generados en OUT con tamaño y timestamp.
# Sirve como índice del apéndice de reproducibilidad.
# Produce:
#   MANIFEST.txt
# =============================================================
manifest = []
for f in sorted(os.listdir(OUT)):
    if f == "MANIFEST.txt": continue
    p = os.path.join(OUT, f)
    manifest.append(f"{f}\t{os.path.getsize(p):>10} bytes\t{datetime.fromtimestamp(os.path.getmtime(p))}")
open(os.path.join(OUT, "MANIFEST.txt"), "w").write("\n".join(manifest))
print("\n".join(manifest))
print(f"\n📁 Total: {len(manifest)} archivos en {OUT}")

AnalisisVariables.ipynb	    156018 bytes	2026-08-27 19:38:01
Carga.ipynb	    526879 bytes	2026-09-01 14:07:04
CargaDatos.ipynb	    127025 bytes	2026-02-14 04:10:03
ELO20250802.txt	     41296 bytes	2026-09-10 02:56:36
ELO20250802_clean.csv	        19 bytes	2026-09-02 16:30:03
ELO20260106.txt	     40825 bytes	2026-08-26 17:08:42
ELO20260902.gdoc	       173 bytes	2026-09-02 19:37:43
ELO20260902.txt	     59400 bytes	2026-09-10 02:56:36
EstadiumsDB (1).csv	     34761 bytes	2026-07-29 17:25:56
EstadiumsDB.csv	     34761 bytes	2026-09-10 02:56:38
EstadiumsDB_normalized.csv	     29571 bytes	2026-07-29 16:08:29
Extra1.ipynb	    213630 bytes	2026-09-04 01:53:45
Extra2.ipynb	   1321851 bytes	2026-09-03 19:03:47
GlickoSoccer-master	      4096 bytes	2026-05-16 10:46:21
Legacy Data	      4096 bytes	2026-04-22 15:20:09
Legacy ipynb py	      4096 bytes	2026-04-22 14:35:23
LimpiezaFinal (1).ipynb	    934754 bytes	2026-08-27 15:05:08
LimpiezaFinal.ipynb	    940857 bytes	2026-09-10 02:56:39
LimpiezaParte

In [10]:
for ds in ["full", "lite"]:
    f = pd.read_csv(os.path.join(BASE, f"glicko_ratings_final_{ds}.csv"))
    print(ds, f["mu"].min(), f["mu"].max(), f["mu"].max()-f["mu"].min())

full 1172.034414719703 1807.1128072246197 635.0783925049168
lite 1153.945999778434 1801.5079242393897 647.5619244609557
